[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-03.ipynb)

# 3주차 실습: 효율적 미세조정(PEFT) - LoRA

**목표.** 사전학습 한국어 분류 모델에 **LoRA 어댑터**를 붙여 미세조정하고, **학습 파라미터 수**가 전체 미세조정 대비 얼마나 줄었는지 출력으로 확인한다. 랭크 값을 하나 바꿔 학습 파라미터 수와 성능이 어떻게 변하는지 본다.

이 실습은 **과제 1**의 출발점이다. 과제 1에서는 이 코드를 그대로 돌리고, 결과를 제공된 비교표에 위치시켜 **도내 소규모 조직이 감당할 메모리·시간 예산** 안에서 선택 근거를 대야 한다.


## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다.


In [1]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers datasets peft accelerate evaluate



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.


### 1-1. 데이터 준비

한국어 영화 리뷰 **감성 분류**(긍정/부정) 데이터셋 NSMC를 사용한다. 원본 저장소(e9t/nsmc)의 파일을 직접 읽고, Colab 무료 티어에서 빠르게 돌도록 소량만 쓴다.


In [2]:
from datasets import load_dataset

# NSMC 원본(e9t/nsmc GitHub)을 CSV로 직접 읽는다. 허브의 nsmc 는 스크립트 데이터셋이라 최신 datasets 에서 지원이 끊겼다
data_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
dataset = load_dataset("csv", data_files=data_url, sep="\t", split="train[:3000]")

# train/eval 로 나눈다
split = dataset.train_test_split(test_size=0.2, seed=42)
train_ds, eval_ds = split["train"], split["test"]

print("train:", len(train_ds), "eval:", len(eval_ds))
print("예시:", train_ds[0]["document"], "->", train_ds[0]["label"])


/private/tmp/claude-501/-Users-macbook-Projects-class-notion/b072bf86-94dd-4a28-b2dc-72062b6c1ba0/scratchpad/deepnlp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train: 2400 eval: 600
예시: 사실여부를 떠나,알고왔던 아더와 너무 매칭이 안돼더라.원탁기사중 실제 검술 최고수는 랜슬롯으로 알고 있는데,트리스탄보다 못하고,싸우는 검술은 마치 중국검술 흉내낸거 같은게;; 그리고 란슬롯이 실제는 쌍검였나?너무 매칭이 안대 하튼 ㅋ기네비어역도 미스. -> 0


### 1-2. 토크나이징


In [3]:
from transformers import AutoTokenizer

model_name = "monologg/koelectra-base-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["document"], padding="max_length", truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["document"])
eval_ds = eval_ds.map(tokenize, batched=True, remove_columns=["document"])
train_ds = train_ds.rename_column("label", "labels")
eval_ds = eval_ds.rename_column("label", "labels")
train_ds.set_format("torch")
eval_ds.set_format("torch")

print(train_ds.column_names)


['id', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']


### 1-3. 사전학습 모델 로딩

2주차에서 로딩한 것과 같은 한국어 분류 모델이다. 먼저 **전체 파라미터 수**를 확인해 둔다.


In [4]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
total_params = sum(p.numel() for p in model.parameters())
print(f"모델 전체 파라미터 수: {total_params:,}")


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 59988.23it/s]


[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your d

모델 전체 파라미터 수: 112,922,882


### 1-4. 전체 미세조정 기준 학습 파라미터 수

**전체 미세조정(full fine-tuning)** 은 모델의 모든 파라미터를 학습시킨다. 즉 학습 파라미터 수 = 전체 파라미터 수다.


In [5]:
full_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 미세조정 시 학습 파라미터 수: {full_trainable:,}")


전체 미세조정 시 학습 파라미터 수: 112,922,882


### 1-5. LoRA 어댑터 붙이기

모델을 **동결(freeze)** 하고, 저랭크 분해 행렬(어댑터)만 학습시킨다. `peft` 라이브러리가 어댑터를 붙이고, 학습 파라미터 수를 세어 준다.


In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,                     # 저랭크 차원
    lora_alpha=8,            # 스케일링 계수 (보통 r 의 2배)
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["query", "value"],
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


trainable params: 739,586 || all params: 113,662,468 || trainable%: 0.6507


### 1-6. LoRA 미세조정 실행

Trainer로 어댑터만 학습시킨다. 몇 분 안에 끝난다.


In [7]:
import numpy as np
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": (preds == labels).mean()}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=100,
    report_to=[],
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print("평가 정확도:", trainer.evaluate())


/private/tmp/claude-501/-Users-macbook-Projects-class-notion/b072bf86-94dd-4a28-b2dc-72062b6c1ba0/scratchpad/deepnlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
150,0.691219,0.687239,0.553333


/private/tmp/claude-501/-Users-macbook-Projects-class-notion/b072bf86-94dd-4a28-b2dc-72062b6c1ba0/scratchpad/deepnlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Accuracy
0.691219,0.687239,150,0.553333


평가 정확도: {'eval_loss': 0.6872387528419495, 'eval_accuracy': 0.5533333333333333}


## 2. 한 지점만 바꿔 보기

아래 셀의 `# TODO` 로 표시된 **한 곳(랭크 r)** 만 바꾸고 다시 실행하세요.

> 바꾸기 전 학습 파라미터 수와 평가 정확도를 먼저 적어 두면 무엇이 달라졌는지 비교할 수 있습니다.


In [8]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# TODO: 랭크 r 값을 바꿔 보세요 (예: 4 -> 8)
lora_config = LoraConfig(
    r=8,                     # TODO 적용: 4 -> 8
    lora_alpha=8,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["query", "value"],
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 52352.40it/s]


[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your d

trainable params: 887,042 || all params: 113,809,924 || trainable%: 0.7794


## 3. 실무: 과제 1 안내

이 실습 코드는 **과제 1**의 밑바탕이다. 과제 1에서 할 일은 다음과 같다.

1. 한국어 데이터셋에서 **PEFT 기법 한 가지**(여기서는 LoRA)를 돌려본다
2. 강의 사이트에 제공된 **비교표**에서 자기 결과(학습 파라미터 수·메모리·시간·정확도)의 위치를 해석한다
3. **도내 소규모 조직이 감당할 메모리·시간 예산**을 조건으로 명시하고, 그 안에서 선택 근거를 댄다

채점은 3단계 부분점수(접근 설계 40 / 동작 40 / 성능·비교 20)로 이뤄진다. **1단계(왜 이 방법을 골랐는지)만 성립해도 부분 점수**를 받는다.

과제 1 상세와 비교표는 `assignments/week-03/` 폴더의 안내문을 참고한다.


## 4. 확인 질문

1. 1-5에서 출력된 LoRA 학습 파라미터 수는 전체 파라미터 수의 몇 퍼센트인가요? 손으로 계산해 보세요.
2. 랭크 r 을 키웠을 때 학습 파라미터 수는 어떻게 변하나요? 왜 그렇게 되는지 저랭크 근사의 의미와 연결지어 설명하세요.
3. 전체 미세조정 대비 LoRA는 메모리·시간·성능 측면에서 각각 무엇이 다른가요?

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.


**답**

1. 739,586 ÷ 112,922,882 ≈ 0.0065, 약 0.65%다. (출력의 0.6507%는 LoRA를 붙인 뒤 전체 113,662,468개 기준이다.)
2. r을 4에서 8로 키우니 학습 파라미터가 739,586개에서 887,042개로 147,456개 늘었다. 학습 파라미터에는 r과 상관없는 분류 헤드(약 592,130개)가 들어 있어서 전체가 두 배가 되지는 않지만, LoRA 행렬 몫만 보면 147,456개에서 294,912개로 정확히 2배다. LoRA는 원래 가중치 옆에 (d×r)과 (r×d) 두 개의 얇은 행렬만 학습하는데 그 크기가 r에 정비례하기 때문이다. r이 크면 저랭크 근사가 표현할 수 있는 변화의 폭도 커진다.
3. 메모리: 학습하는 파라미터가 0.65%뿐이라 기울기와 옵티마이저 상태가 훨씬 작다. 시간: 갱신할 파라미터가 적어 한 걸음이 빠르고, 과제마다 작은 어댑터만 저장하면 된다. 성능: 보통 전체 미세조정과 비슷하거나 조금 낮다. 이번 실습은 데이터 2,400개·1에폭이라 정확도가 55~57%에 그쳤고, 전체 미세조정과 직접 비교하지는 않았다.


## 5. 제출

이 노트북은 과제 1 이전의 **연습용**이다. 과제 1 답안은 별도 안내(비교표 포함)를 따라 제출한다.

연습 결과를 기록하려면 2주차와 같은 방법으로 `assignments/week-03/<내 학번>/` 에 올리면 된다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.
